# OCR de Kanjis N2-N5 com YOLOv8

Este notebook foi projetado para rodar no **Kaggle** ou **Google Colab** (com GPU ativada).

Ele orquestra o pipeline completo do projeto:
1. Instala dependências
2. Clona o repositório
3. Baixa fontes CJK
4. Gera o dataset sintético balanceado (~69k imagens)
5. Treina o YOLOv8n por 50 épocas
6. Disponibiliza os pesos treinados para download
7. Executa inferência em imagens de teste

## 0. Verificar GPU

In [ ]:
import torch
print(f'CUDA disponível: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('AVISO: GPU não encontrada. Treinamento será muito mais lento.')

## 1. Instalar Dependências e Clonar Repositório

In [ ]:
!pip install -q ultralytics opencv-python-headless Pillow tqdm

import os
REPO_NAME = 'OCR-de-kanjis-N2-N5-com-YoloV8'
GITHUB_URL = 'https://github.com/Thomaz332/OCR-de-kanjis-N2-N5-com-YoloV8.git'

if os.path.exists(REPO_NAME):
    print(f'Repositório {REPO_NAME} já existe. Atualizando...')
    os.system(f'cd {REPO_NAME} && git pull')
else:
    os.system(f'git clone {GITHUB_URL}')

%cd {REPO_NAME}
print('Diretório atual:', os.getcwd())

## 2. Baixar Fontes CJK

In [ ]:
!python src/data/download_fonts.py

import glob
fontes = glob.glob('assets/fonts/*')
print(f'Fontes disponíveis: {[f.split("/")[-1] for f in fontes]}')

## 3. Gerar Dataset Sintético N2-N5

**Estratégia:** Dataset 100% sintético e balanceado — cada kanji recebe o mesmo número de
imagens, eliminando o viés de frequência natural dos mangás.

- ~690 classes (689 kanjis N2-N5 + UNKNOWN_N1)
- 100 imagens/classe = ~69.000 imagens no total
- Split automático 80/20 (treino/val)
- Augmentations: distorção elástica, screentone, textura de papel, etc.

In [ ]:
!python src/data/generate_synthetic_images.py

import os
train_imgs = len(os.listdir('data/synthetic/images/train'))
val_imgs   = len(os.listdir('data/synthetic/images/val'))
print(f'Imagens de treino: {train_imgs}')
print(f'Imagens de val:    {val_imgs}')
print(f'Total:             {train_imgs + val_imgs}')

## 4. Treinar o Modelo YOLOv8n

In [ ]:
import os
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

data_path = os.path.abspath('data/synthetic/data.yaml')
print(f'Usando configuração: {data_path}')

results = model.train(
    data=data_path,
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,       # GPU
    project='yolo_kanji',
    name='n2_n5_model',
    plots=True,
    verbose=True,
)
print('Treinamento concluído!')

## 5. Download dos Pesos Treinados

In [ ]:
from IPython.display import FileLink, display
import glob, os

train_dirs = glob.glob('yolo_kanji/n2_n5_model*')
if not train_dirs:
    train_dirs = glob.glob('runs/detect/train*')

if train_dirs:
    latest = max(train_dirs, key=os.path.getmtime)
    weights = f'{latest}/weights/best.pt'
    if os.path.exists(weights):
        print(f'Modelo treinado: {weights}')
        print(f'Tamanho: {os.path.getsize(weights) / 1e6:.1f} MB')
        display(FileLink(weights))
    else:
        print('Pesos ainda não gerados.')
else:
    print('Nenhum treinamento encontrado.')

## 6. Testar o Modelo em Imagens de Mangá

Faça upload de imagens de mangá na pasta `manga_test/` e rode a inferência.

In [ ]:
from ultralytics import YOLO
import glob, os, cv2
import matplotlib.pyplot as plt
from IPython.display import FileLink, display

test_folder = 'manga_test/'
os.makedirs(test_folder, exist_ok=True)

imagens = glob.glob(f'{test_folder}/*.jpg') + glob.glob(f'{test_folder}/*.png')

if not imagens:
    print(f'Nenhuma imagem em {test_folder}.')
    print('Faça upload de imagens de mangá nesta pasta e rode a célula novamente.')
else:
    print(f'{len(imagens)} imagem(ns) encontrada(s).')

    # Carregar modelo treinado
    train_dirs = glob.glob('yolo_kanji/n2_n5_model*') or glob.glob('runs/detect/train*')
    latest = max(train_dirs, key=os.path.getmtime)
    model = YOLO(f'{latest}/weights/best.pt')

    # Inferência
    results = model(test_folder, conf=0.25, save=True, project='resultados_manga', name='predicoes')

    # Exibir primeiro resultado
    pred_imgs = glob.glob('resultados_manga/predicoes/*.jpg')
    if pred_imgs:
        img = cv2.cvtColor(cv2.imread(pred_imgs[0]), cv2.COLOR_BGR2RGB)
        plt.figure(figsize=(12, 12))
        plt.imshow(img)
        plt.axis('off')
        plt.title('Detecções N2-N5 (primeiro resultado)')
        plt.show()

    # Zipar resultados
    os.system('zip -q -r resultados_manga.zip resultados_manga/')
    print('Download dos resultados:')
    display(FileLink('resultados_manga.zip'))